# Loading Data from Github

In [447]:
import pandas as pd
import numpy as np
import ast

In [448]:
"""from getpass import getpass
import os

# Prompt for GitHub personal access token
token = getpass("Enter your GitHub personal access token: ")

# Set up GitHub authentication
repo_url = "https://github.com/RawJat/FIFS_IPL.git"
repo_url_with_auth = repo_url.replace("https://", f"https://{token}@")

# Clone the repo
!git clone {repo_url_with_auth}"""

'from getpass import getpass\nimport os\n\n# Prompt for GitHub personal access token\ntoken = getpass("Enter your GitHub personal access token: ")\n\n# Set up GitHub authentication\nrepo_url = "https://github.com/RawJat/FIFS_IPL.git"\nrepo_url_with_auth = repo_url.replace("https://", f"https://{token}@")\n\n# Clone the repo\n!git clone {repo_url_with_auth}'

In [449]:
import os
os.listdir("FIFS_IPL")

['scripts', '.gitignore', '.git', 'notebooks', 'data', 'docs']

In [450]:
df = pd.read_csv('/content/FIFS_IPL/data/bbb/cricsheet_ipl_bbb.csv')

In [451]:
people = pd.read_csv('/content/FIFS_IPL/data/register_data/people.csv')

In [452]:
mapping = pd.read_excel('/content/cricsheet_cricbuzz_mapping_updated.xlsx')

# Cleaning data

Cleaning Null batter role, bat style, bowler role, bowl style

In [453]:
name_to_role = dict(zip(mapping['cricsheet_name'], mapping['role']))
batter_to_style = dict(zip(mapping['cricsheet_name'], mapping['bat_style']))
bowler_to_style = dict(zip(mapping['cricsheet_name'], mapping['bowl_style']))

df['batter_role'] = df['batter'].map(name_to_role)
df['bowler_role'] = df['bowler'].map(name_to_role)
df['batter_bat_style'] = df['batter'].map(batter_to_style)
df['bowler_bowl_style'] = df['bowler'].map(bowler_to_style)

Dropping unnecessary columns

In [454]:
df.drop(columns = ['other_wicket_type', 'other_player_dismissed', 'batter_role', 'batter_bat_style', 'bowler_role', 'bowler_bowl_style', 'penalty'], inplace=True)

Fill null values

In [455]:
df['wides'] = df['wides'].fillna(0)
df['noballs'] = df['noballs'].fillna(0)
df['byes'] = df['byes'].fillna(0)
df['legbyes'] = df['legbyes'].fillna(0)

Create new columns

In [456]:
df['batter_balls_faced'] = np.where(df['wides']==0, 1, 0)
df['bowler_runs'] = df['runs_off_bat'] + df['wides'] + df['noballs']

Resolving player name for consistency

In [457]:
people = people[['unique_name', 'key_cricinfo']]

In [458]:
csheet_to_cbuzz = dict(zip(mapping['cricsheet_name'], mapping['cricbuzz_name']))
people.loc[:, 'cricbuzz_name'] = people['unique_name'].map(csheet_to_cbuzz)

<ipython-input-458-d8028eb9024b>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  people.loc[:, 'cricbuzz_name'] = people['unique_name'].map(csheet_to_cbuzz)


In [459]:
# Create a dictionary mapping unique_name to key_cricinfo
name_to_id = dict(zip(people['unique_name'], people['key_cricinfo']))
# Map batter and bowler names to their respective IDs using the dictionary
df['batter_id'] = df['batter'].map(name_to_id)
df['bowler_id'] = df['bowler'].map(name_to_id)

In [460]:
mapping = mapping[['cricsheet_name', 'cricbuzz_name']]

In [461]:
# Create a dictionary mapping unique_name to key_cricinfo
csheet_to_cbuzz = dict(zip(mapping['cricsheet_name'], mapping['cricbuzz_name']))
# Map batter and bowler names to their respective IDs using the dictionary
df['batter'] = df['batter'].map(csheet_to_cbuzz)
df['bowler'] = df['bowler'].map(csheet_to_cbuzz)
df['player_dismissed'] = df['player_dismissed'].map(csheet_to_cbuzz)

In [462]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108850 entries, 0 to 108849
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   match_id            108850 non-null  int64  
 1   start_date          108850 non-null  object 
 2   venue               108850 non-null  object 
 3   innings             108850 non-null  int64  
 4   ball                108850 non-null  float64
 5   batting_team        108850 non-null  object 
 6   bowling_team        108850 non-null  object 
 7   batter              108850 non-null  object 
 8   bowler              108850 non-null  object 
 9   runs_off_bat        108850 non-null  int64  
 10  extras              108850 non-null  int64  
 11  wides               108850 non-null  float64
 12  noballs             108850 non-null  float64
 13  byes                108850 non-null  float64
 14  legbyes             108850 non-null  float64
 15  wicket_type         5370 non-null 

# Batting Scorecard

In [463]:
def process_batting_scorecard_from_kaggle(df):
    if df.empty:
        return pd.DataFrame()

    batting_stats = {}

    for _, ball in df.iterrows():
        match_id = ball['match_id']
        match_date = ball['start_date']
        venue = ball['venue']

        striker = ball['batter_id']
        batsman_name = ball['batter']
        bowler_name = ball['bowler']
        batting_team = ball['batting_team']
        bowling_team = ball['bowling_team']

        key = (match_id, batting_team, striker)
        if key not in batting_stats:
            batting_stats[key] = {
                'batsman_name': batsman_name,
                'bowler_name': "",
                'runs': 0,
                'balls': 0,
                'fours': 0,
                'sixes': 0,
                'isOut': False,
                'wicketType': 'not out',
                'innings': ball['innings'],
                'p_bat_id': striker,
                'p_bowl_id': None,
                'bat_team': batting_team,
                'bowl_team': bowling_team,
                'date': match_date,
                'venue': venue
            }

        stats = batting_stats[key]

        runs = ball['runs_off_bat']
        stats['runs'] += runs

        # Count boundaries
        if runs == 4:
            stats['fours'] += 1
        elif runs == 6:
            stats['sixes'] += 1

        # Count balls faced (excluding wides and no balls)
        if ball['batter_balls_faced'] == 1:
            stats['balls'] += 1

        # Check if batsman is out
        if pd.notna(ball['wicket_type']) and ball['player_dismissed'] == batsman_name:
            stats['isOut'] = True
            stats['wicketType'] = ball['wicket_type']
            stats['bowler_name'] = bowler_name
            stats['p_bowl_id'] = ball['bowler_id']

        batting_stats[key] = stats

    results = []
    for (match_id, team, player), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'match_id': match_id,
            'date': stats['date'],
            'venue': stats['venue'],
            'inns': stats['innings'],
            'batter_id': player,
            'batter': stats['batsman_name'],
            'bowler': stats['bowler_name'],
            'bowler_id': stats['p_bowl_id'],
            'runs': stats['runs'],
            'balls': stats['balls'],
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'sr': round(strike_rate, 2),
            'isOut': stats['isOut'],
            'wicketType': stats['wicketType'],
            'bat_team': stats['bat_team'],
            'bowl_team': stats['bowl_team']
        })

    return pd.DataFrame(results)

batting_card=process_batting_scorecard_from_kaggle(df)
batting_card.sort_values(by=['date', 'match_id', 'inns'], inplace=True)
batting_card['bowler'] = batting_card['bowler'].replace("", pd.NA)

In [464]:
# Update team names in team1_name and team2_name columns
batting_card['bat_team'] = batting_card['bat_team'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})
batting_card['bowl_team'] = batting_card['bowl_team'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})

In [465]:
batting_card['batter_id'] = batting_card['batter_id'].astype(int)

In [466]:
batting_card[batting_card['match_id'] == 335985]

,match_id,date,venue,inns,batter_id,batter,bowler,bowler_id,runs,balls,fours,sixes,sr,isOut,wicketType,bat_team,bowl_team
59,335985,2008-04-20,Wankhede Stadium,1,7502,Luke Ronchi,Zaheer Khan,30102.0,8,10,2,0,80.00,True,caught,Mumbai Indians,Royal Challengers Bangalore
60,335985,2008-04-20,Wankhede Stadium,1,49209,Sanath Jayasuriya,Zaheer Khan,30102.0,29,16,5,1,181.25,True,run out,Mumbai Indians,Royal Challengers Bangalore
61,335985,2008-04-20,Wankhede Stadium,1,8002,Dominic Thornely,Zaheer Khan,30102.0,0,5,0,0,0.00,True,retired hurt,Mumbai Indians,Royal Challengers Bangalore
62,335985,2008-04-20,Wankhede Stadium,1,35582,Robin Uthappa,Balachandra Akhil,26206.0,48,38,5,1,126.32,True,stumped,Mumbai Indians,Royal Challengers Bangalore
63,335985,2008-04-20,Wankhede Stadium,1,34282,Pinal Shah,Balachandra Akhil,26206.0,19,23,2,0,82.61,True,caught,Mumbai Indians,Royal Challengers Bangalore
64,335985,2008-04-20,Wankhede Stadium,1,32091,Manu Nayyar,Praveen Kumar,30732.0,20,14,1,1,142.86,True,caught,Mumbai Indians,Royal Challengers Bangalore
65,335985,2008-04-20,Wankhede Stadium,1,46774,Shaun Pollock,Zaheer Khan,30102.0,28,12,3,2,233.33,True,bowled,Mumbai Indians,Royal Challengers Bangalore
66,335985,2008-04-20,Wankhede Stadium,1,29264,Harbhajan Singh,<NA>,NaN,2,2,0,0,100.00,False,not out,Mumbai Indians,Royal Challengers Bangalore
67,335985,2008-04-20,Wankhede Stadium,2,51469,Shivnarine Chanderpaul,Dhawal Kulkarni,277955.0,16,11,3,0,145.45,True,caught,Royal Challengers Bangalore,Mumbai Indians
68,335985,2008-04-20,Wankhede Stadium,2,28114,Rahul Dravid,Harbhajan Singh,29264.0,32,33,4,0,96.97,True,caught and bowled,Royal Challengers Bangalore,Mumbai Indians


In [467]:
batting_card.to_csv('cricsheet_ipl_mbm_bat.csv', index=False)

# Bowling Card

In [468]:
df.columns

Index(['match_id', 'start_date', 'venue', 'innings', 'ball', 'batting_team',
       'bowling_team', 'batter', 'bowler', 'runs_off_bat', 'extras', 'wides',
       'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed',
       'batter_balls_faced', 'bowler_runs', 'batter_id', 'bowler_id'],
      dtype='object')

In [469]:
df['bowlruns'] = df['runs_off_bat'] + df['wides'] + df['noballs']

In [470]:
df['wicket_type'].unique()

array([nan, 'caught', 'bowled', 'run out', 'lbw', 'retired hurt',
       'stumped', 'caught and bowled', 'hit wicket',
       'obstructing the field'], dtype=object)

In [471]:
import pandas as pd
import numpy as np

def calculate_bowling_card(df):
    # Ensure necessary columns are of the right type
    df = df.copy()

    # Convert columns to appropriate types
    df['noballs'] = df['noballs'].fillna(0).astype(float)
    df['wides'] = df['wides'].fillna(0).astype(float)
    df['wicket_type'] = df['wicket_type'].fillna('')
    df['runs_off_bat'] = df['runs_off_bat'].fillna(0).astype(float)
    df['bowler_runs'] = df['bowler_runs'].fillna(0).astype(float)

    # Ensure ball column is string
    df['ball'] = df['ball'].astype(str)

    # Create legal delivery indicator
    df['legal_delivery'] = (df['noballs'] == 0) & (df['wides'] == 0)

    # Create valid wicket indicator (excluding run outs)
    df['valid_wicket'] = (df['wicket_type'].notna()) & (df['wicket_type'] != '') & (df['wicket_type'] != 'run out')

    # Group by match, bowler, bowler ID, innings
    bowling_card = df.groupby(['match_id', 'bowler', 'bowler_id', 'bowling_team', 'batting_team', 'innings'])

    # Compute required statistics
    summary = bowling_card.agg(
        overs=('legal_delivery', lambda x: f"{x.sum() // 6}.{x.sum() % 6}"),  # Overs in x.y format
        balls=('ball', 'count'),  # Total deliveries (including wides and no-balls)
        wickets=('valid_wicket', 'sum'),  # Count of valid wickets (excluding run outs)
        runs=('bowler_runs', 'sum'),  # Total runs conceded
        lbw_or_bowled=('wicket_type', lambda x: ((x == 'lbw') | (x == 'bowled')).sum()),
        dot_balls=('runs_off_bat', lambda x: (x == 0).sum())  # Dot balls where no runs were scored
    ).reset_index()

    # Calculate economy rate
    def calculate_economy(row):
        # Handle cases with 0 overs
        if row['overs'] == '0.0':
            return 0.0  # Or np.inf, depending on how you want to handle this

        # Calculate total overs as a float
        overs_parts = row['overs'].split('.')
        total_overs = int(overs_parts[0]) + int(overs_parts[1]) / 6

        # Prevent division by zero
        return row['runs'] / total_overs if total_overs > 0 else 0.0

    summary['economy'] = summary.apply(calculate_economy, axis=1)

    # Calculate maidens
    maiden_overs = df.groupby(['match_id', 'bowler', 'bowler_id', 'innings', df['ball'].str.split('.').str[0].astype(int)]).agg(
        over_runs=('runs_off_bat', 'sum')
    ).reset_index()

    maidens = maiden_overs.groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['over_runs'].apply(lambda x: (x == 0).sum()).reset_index()
    summary = summary.merge(maidens, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')
    summary.rename(columns={'over_runs': 'maidens'}, inplace=True)

    # Add venue and date
    match_info = df.groupby('match_id').agg(
        venue=('venue', 'first'),
        date=('start_date', 'first')
    ).reset_index()
    summary = summary.merge(match_info, on='match_id', how='left')

    # Determine the first over bowled by each player in each innings
    first_over = df[df['legal_delivery']].groupby(['match_id', 'bowler', 'bowler_id', 'innings'])['ball'].apply(
        lambda x: min(int(str(val).split('.')[0]) for val in x)
    ).reset_index()

    summary = summary.merge(first_over, on=['match_id', 'bowler', 'bowler_id', 'innings'], how='left')

    # Sort by match, innings, and first over
    summary = summary.sort_values(by=['match_id', 'innings', 'ball'])

    summary.drop(columns=['ball'], inplace=True)

    return summary

# Example usage
bowling_card = calculate_bowling_card(df)

In [472]:
bowling_card

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date
7,335982,Praveen Kumar,30732.0,Royal Challengers Bangalore,Kolkata Knight Riders,1,4.0,25,0,38.0,0,14,9.500000,1,M Chinnaswamy Stadium,2008-04-18
10,335982,Zaheer Khan,30102.0,Royal Challengers Bangalore,Kolkata Knight Riders,1,4.0,24,1,38.0,0,8,9.500000,0,M Chinnaswamy Stadium,2008-04-18
1,335982,Ashley Noffke,6938.0,Royal Challengers Bangalore,Kolkata Knight Riders,1,4.0,25,1,40.0,0,8,10.000000,0,M Chinnaswamy Stadium,2008-04-18
5,335982,Jacques Kallis,45789.0,Royal Challengers Bangalore,Kolkata Knight Riders,1,4.0,25,1,48.0,0,10,12.000000,0,M Chinnaswamy Stadium,2008-04-18
9,335982,Sunil Joshi,29725.0,Royal Challengers Bangalore,Kolkata Knight Riders,1,3.0,18,0,26.0,0,4,8.666667,0,M Chinnaswamy Stadium,2008-04-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5455,734049,Mitchell Johnson,6033.0,Kings XI Punjab,Kolkata Knight Riders,2,4.0,26,2,41.0,0,10,10.250000,0,M Chinnaswamy Stadium,2014-06-01
5454,734049,Lakshmipathy Balaji,26829.0,Kings XI Punjab,Kolkata Knight Riders,2,4.0,24,0,41.0,0,6,10.250000,0,M Chinnaswamy Stadium,2014-06-01
5457,734049,Parvinder Awana,323131.0,Kings XI Punjab,Kolkata Knight Riders,2,3.3,21,0,43.0,0,3,12.285714,0,M Chinnaswamy Stadium,2014-06-01
5453,734049,Karnvir Singh,322535.0,Kings XI Punjab,Kolkata Knight Riders,2,4.0,27,4,54.0,0,12,13.500000,0,M Chinnaswamy Stadium,2014-06-01


In [473]:
bowling_card[bowling_card['match_id'] == 734041]

,match_id,bowler,bowler_id,bowling_team,batting_team,innings,overs,balls,wickets,runs,lbw_or_bowled,dot_balls,economy,maidens,venue,date
5408,734041,Corey Anderson,277662.0,Mumbai Indians,Rajasthan Royals,1,2.0,12,0,14.0,0,7,7.000000,0,Wankhede Stadium,2014-05-25
5412,734041,Jasprit Bumrah,625383.0,Mumbai Indians,Rajasthan Royals,1,4.0,25,1,30.0,0,12,7.500000,1,Wankhede Stadium,2014-05-25
5415,734041,Pragyan Ojha,32130.0,Mumbai Indians,Rajasthan Royals,1,4.0,25,0,31.0,0,11,7.750000,0,Wankhede Stadium,2014-05-25
5410,734041,Harbhajan Singh,29264.0,Mumbai Indians,Rajasthan Royals,1,4.0,24,1,43.0,0,7,10.750000,0,Wankhede Stadium,2014-05-25
5418,734041,Shreyas Gopal,344580.0,Mumbai Indians,Rajasthan Royals,1,3.0,18,1,36.0,0,3,12.000000,0,Wankhede Stadium,2014-05-25
5414,734041,Kieron Pollard,230559.0,Mumbai Indians,Rajasthan Royals,1,3.0,19,1,33.0,0,5,11.000000,0,Wankhede Stadium,2014-05-25
5411,734041,James Faulkner,270484.0,Rajasthan Royals,Mumbai Indians,2,3.4,23,0,54.0,0,6,14.727273,0,Wankhede Stadium,2014-05-25
5409,734041,Dhawal Kulkarni,277955.0,Rajasthan Royals,Mumbai Indians,2,3.0,20,2,42.0,0,9,14.000000,0,Wankhede Stadium,2014-05-25
5417,734041,Shane Watson,8180.0,Rajasthan Royals,Mumbai Indians,2,2.0,12,0,33.0,0,2,16.500000,0,Wankhede Stadium,2014-05-25
5413,734041,Kevon Cooper,371814.0,Rajasthan Royals,Mumbai Indians,2,4.0,24,2,38.0,1,9,9.500000,0,Wankhede Stadium,2014-05-25


In [474]:
bowling_card.rename(columns={'batting_team':'bat_team', 'bowling_team':'bowl_team', 'innings':'inns', 'economy':'eco', 'dot_balls':'dots'}, inplace=True)

In [475]:
bowling_card.columns

Index(['match_id', 'bowler', 'bowler_id', 'bowl_team', 'bat_team', 'inns',
       'overs', 'balls', 'wickets', 'runs', 'lbw_or_bowled', 'dots', 'eco',
       'maidens', 'venue', 'date'],
      dtype='object')

In [476]:
bowling_card.to_csv('cricsheet_ipl_mbm_bowl.csv', index=False)